In [1]:
!pip install openpyxl


[notice] A new release of pip is available: 23.2.1 -> 24.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install dash dash-bootstrap-components matplotlib pandas geopandas plotly jupyter-dash



[notice] A new release of pip is available: 23.2.1 -> 24.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import geopandas as gpd
from dash import Dash, dcc, html
import dash_bootstrap_components as dbc
from dash.dependencies import Input, Output
import plotly.express as px

# Load the Excel file
file_path = 'Copy of HEAT_Tables_20.06.24_CPedits.xlsx'
xlsx_file = pd.read_excel(file_path, sheet_name=['RP1', 'Abj_outputs', 'Jhb_outputs', 'RP1_Countries'])

# Extract each DataFrame from the dictionary
df_rp1 = xlsx_file['RP1']
df_abj = xlsx_file['Abj_outputs']
df_jhb = xlsx_file['Jhb_outputs']
df_rp1_countries = xlsx_file['RP1_Countries']

# Define the color map and stage order to match the actual column names in df_rp1_countries
color_map = {
    '1st invite': '#1f77b4',
    '2nd invite': '#ff7f0e',
    '3rd or more invite': '#b8daff',
    'Interested': '#7f7f7f',
    'Follow up on interest': '#2ca02c',
    'Study documents received': '#d62728',
    'Eligibility confirmed': '#bcbd22',
    'Entering DTA': '#17becf',
    'DTA complete': '#9467bd',
    'Ineligible': '#8c564b',
    'Declined participation': '#ff9896',
    'Data unavailable': '#c5b0d5'
}
stage_order = [
    '1st invite', '2nd invite', '3rd or more invite', 'Interested',
    'Follow up on interest', 'Study documents received', 'Eligibility confirmed',
    'Entering DTA', 'DTA complete', 'Ineligible', 'Declined participation', 'Data unavailable'
]

# Convert the month columns in each DataFrame to a datetime format
def map_month_year(df, start_year=2023):
    month_map = {}
    encountered_dec = False  # Flag to indicate if we've encountered 'Dec'
    months_in_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    for column in df.columns:
        if any(month in column for month in months_in_order):
            base_month_name = ''.join(filter(str.isalpha, column))
            if base_month_name == 'Jan' and encountered_dec:
                start_year += 1
            if base_month_name == 'Dec':
                encountered_dec = True
            month_map[column] = f'{base_month_name} {start_year}'
    return df.rename(columns=month_map)

df_rp1 = map_month_year(df_rp1)
df_abj = map_month_year(df_abj)
df_jhb = map_month_year(df_jhb)

# Convert column names to datetime
def convert_column_to_datetime(df):
    new_columns = []
    for col in df.columns:
        if col == 'Stage':
            new_columns.append(col)
        else:
            date_str = col.strip() + ' 1'
            new_columns.append(pd.to_datetime(date_str, format='%b %Y %d', errors='coerce'))
    df.columns = new_columns

for dataframe in [df_rp1, df_abj, df_jhb]:
    convert_column_to_datetime(dataframe)

# Ensure the column names in df_rp1_countries match the stage_order
df_rp1_countries.columns = df_rp1_countries.columns.str.strip()  # Remove any leading/trailing whitespace

# Prepare df_rp1_countries for plotting
df_rp1_countries.set_index('Study site', inplace=True)
df_rp1_countries = df_rp1_countries[stage_order].fillna(0)
df_rp1_countries = df_rp1_countries[df_rp1_countries.sum(axis=1) > 0]
df_rp1_countries['Study site'] = df_rp1_countries.index

# Function to plot the main stacked bar chart
def plot_stacked_bar_chart(df, title, last_n_months=8, color_map=None, stage_order=None):
    df = df.set_index('Stage').reindex(stage_order).fillna(0).reset_index()
    excluded_df = df[df['Stage'] == 'Ineligible/declined participation/data currently unavailable']
    df = df[df['Stage'] != 'Ineligible/declined participation/data currently unavailable']
    stages_df = df[~df['Stage'].str.contains("Total")]
    transposed_df = stages_df.set_index('Stage').transpose()
    transposed_df = transposed_df.iloc[-last_n_months:]

    # Debugging print statements
    print("Transposed DataFrame for", title)
    print(transposed_df.head())

    fig = px.bar(transposed_df, barmode='stack', title=title,
                 color_discrete_map=color_map)
    fig.update_layout(xaxis_title='Month', yaxis_title='Number of Studies')
    fig.update_traces(texttemplate='%{text:.2s}', textposition='inside')
    fig.update_xaxes(tickformat='%b %Y')
    return fig

# Create the app
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = html.Div([
    dbc.Row([
        dbc.Col(html.Div([
            dcc.Graph(id='rp1-bar-chart')
        ]), width=12)
    ]),
    dbc.Row([
        dbc.Col(html.Div([
            dcc.Graph(id='abj-bar-chart')
        ]), width=6),
        dbc.Col(html.Div([
            dcc.Graph(id='jhb-bar-chart')
        ]), width=6)
    ]),
    dbc.Row([
        dbc.Col(html.Div([
            dcc.Graph(id='rp1-countries-bar-chart')
        ]), width=12)
    ])
])

@app.callback(
    [Output('rp1-bar-chart', 'figure'),
     Output('abj-bar-chart', 'figure'),
     Output('jhb-bar-chart', 'figure'),
     Output('rp1-countries-bar-chart', 'figure')],
    [Input('rp1-bar-chart', 'id'),
     Input('abj-bar-chart', 'id'),
     Input('jhb-bar-chart', 'id'),
     Input('rp1-countries-bar-chart', 'id')]
)
def update_charts(*args):
    rp1_fig = plot_stacked_bar_chart(df_rp1, 'RP1 Progress', last_n_months=8, color_map=color_map, stage_order=stage_order)
    abj_fig = plot_stacked_bar_chart(df_abj, 'Abidjan Progress', last_n_months=8, color_map=color_map, stage_order=stage_order)
    jhb_fig = plot_stacked_bar_chart(df_jhb, 'Johannesburg Progress', last_n_months=8, color_map=color_map, stage_order=stage_order)
    
    # Plot the RP1 countries stacked bar chart
    melted_df = df_rp1_countries.melt(id_vars='Study site', value_vars=stage_order)
    print("Melted DataFrame for RP1 Countries")
    print(melted_df.head())
    fig = px.bar(melted_df, x='Study site', y='value', color='variable', title='Number of Studies in Relevant African Countries (RP1)',
                 color_discrete_map=color_map, barmode='stack')
    fig.update_layout(xaxis_title='Country', yaxis_title='Number of Studies')
    fig.update_traces(texttemplate='%{text:.2s}', textposition='inside')
    fig.update_xaxes(tickangle=-45)
    
    countries_fig = fig
    
    return rp1_fig, abj_fig, jhb_fig, countries_fig

if __name__ == '__main__':
    app.run_server(mode='inline')


In [4]:
import pandas as pd
import geopandas as gpd
from dash import Dash, dcc, html
import dash_bootstrap_components as dbc
from dash.dependencies import Input, Output
import plotly.express as px

# Load the Excel file
file_path = 'Copy of HEAT_Tables_20.06.24_CPedits.xlsx'
xlsx_file = pd.read_excel(file_path, sheet_name=['RP1', 'Abj_outputs', 'Jhb_outputs', 'RP1_Countries'])

# Extract each DataFrame from the dictionary
df_rp1 = xlsx_file['RP1']
df_abj = xlsx_file['Abj_outputs']
df_jhb = xlsx_file['Jhb_outputs']
df_rp1_countries = xlsx_file['RP1_Countries']

# Inspect the original data
print("RP1 DataFrame:")
print(df_rp1.head())
print("\nAbidjan Outputs DataFrame:")
print(df_abj.head())
print("\nJohannesburg Outputs DataFrame:")
print(df_jhb.head())
print("\nRP1 Countries DataFrame:")
print(df_rp1_countries.head())

# Define the color map and stage order to match the actual column names in df_rp1_countries
color_map = {
    '1st invite': '#1f77b4',
    '2nd invite': '#ff7f0e',
    '3rd or more invite': '#b8daff',
    'Interested': '#7f7f7f',
    'Follow up on interest': '#2ca02c',
    'Study documents received': '#d62728',
    'Eligibility confirmed': '#bcbd22',
    'Entering DTA': '#17becf',
    'DTA complete': '#9467bd',
    'Ineligible': '#8c564b',
    'Declined participation': '#ff9896',
    'Data unavailable': '#c5b0d5'
}
stage_order = [
    '1st invite', '2nd invite', '3rd or more invite', 'Interested',
    'Follow up on interest', 'Study documents received', 'Eligibility confirmed',
    'Entering DTA', 'DTA complete', 'Ineligible', 'Declined participation', 'Data unavailable'
]

# Convert the month columns in each DataFrame to a datetime format
def map_month_year(df, start_year=2023):
    month_map = {}
    encountered_dec = False  # Flag to indicate if we've encountered 'Dec'
    months_in_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    for column in df.columns:
        if any(month in column for month in months_in_order):
            base_month_name = ''.join(filter(str.isalpha, column))
            if base_month_name == 'Jan' and encountered_dec:
                start_year += 1
            if base_month_name == 'Dec':
                encountered_dec = True
            month_map[column] = f'{base_month_name} {start_year}'
    return df.rename(columns=month_map)

df_rp1 = map_month_year(df_rp1)
df_abj = map_month_year(df_abj)
df_jhb = map_month_year(df_jhb)

# Convert column names to datetime
def convert_column_to_datetime(df):
    new_columns = []
    for col in df.columns:
        if col == 'Stage':
            new_columns.append(col)
        else:
            date_str = col.strip() + ' 1'
            new_columns.append(pd.to_datetime(date_str, format='%b %Y %d', errors='coerce'))
    df.columns = new_columns

for dataframe in [df_rp1, df_abj, df_jhb]:
    convert_column_to_datetime(dataframe)

# Ensure the column names in df_rp1_countries match the stage_order
df_rp1_countries.columns = df_rp1_countries.columns.str.strip()  # Remove any leading/trailing whitespace

# Prepare df_rp1_countries for plotting
df_rp1_countries.set_index('Study site', inplace=True)
df_rp1_countries = df_rp1_countries[stage_order].fillna(0)
df_rp1_countries = df_rp1_countries[df_rp1_countries.sum(axis=1) > 0]
df_rp1_countries['Study site'] = df_rp1_countries.index

# Inspect the transformed data
print("\nTransformed RP1 DataFrame:")
print(df_rp1.head())
print("\nTransformed Abidjan Outputs DataFrame:")
print(df_abj.head())
print("\nTransformed Johannesburg Outputs DataFrame:")
print(df_jhb.head())
print("\nTransformed RP1 Countries DataFrame:")
print(df_rp1_countries.head())

# Function to plot the main stacked bar chart
def plot_stacked_bar_chart(df, title, last_n_months=8, color_map=None, stage_order=None):
    df = df.set_index('Stage').reindex(stage_order).fillna(0).reset_index()
    excluded_df = df[df['Stage'] == 'Ineligible/declined participation/data currently unavailable']
    df = df[df['Stage'] != 'Ineligible/declined participation/data currently unavailable']
    stages_df = df[~df['Stage'].str.contains("Total")]
    transposed_df = stages_df.set_index('Stage').transpose()
    transposed_df = transposed_df.iloc[-last_n_months:]

    # Debugging print statements
    print("Transposed DataFrame for", title)
    print(transposed_df.head())

    fig = px.bar(transposed_df, barmode='stack', title=title,
                 color_discrete_map=color_map)
    fig.update_layout(xaxis_title='Month', yaxis_title='Number of Studies')
    fig.update_traces(texttemplate='%{text:.2s}', textposition='inside')
    fig.update_xaxes(tickformat='%b %Y')
    return fig

# Create the app
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = html.Div([
    dbc.Row([
        dbc.Col(html.Div([
            dcc.Graph(id='rp1-bar-chart')
        ]), width=12)
    ]),
    dbc.Row([
        dbc.Col(html.Div([
            dcc.Graph(id='abj-bar-chart')
        ]), width=6),
        dbc.Col(html.Div([
            dcc.Graph(id='jhb-bar-chart')
        ]), width=6)
    ]),
    dbc.Row([
        dbc.Col(html.Div([
            dcc.Graph(id='rp1-countries-bar-chart')
        ]), width=12)
    ])
])

@app.callback(
    [Output('rp1-bar-chart', 'figure'),
     Output('abj-bar-chart', 'figure'),
     Output('jhb-bar-chart', 'figure'),
     Output('rp1-countries-bar-chart', 'figure')],
    [Input('rp1-bar-chart', 'id'),
     Input('abj-bar-chart', 'id'),
     Input('jhb-bar-chart', 'id'),
     Input('rp1-countries-bar-chart', 'id')]
)
def update_charts(*args):
    rp1_fig = plot_stacked_bar_chart(df_rp1, 'RP1 Progress', last_n_months=8, color_map=color_map, stage_order=stage_order)
    abj_fig = plot_stacked_bar_chart(df_abj, 'Abidjan Progress', last_n_months=8, color_map=color_map, stage_order=stage_order)
    jhb_fig = plot_stacked_bar_chart(df_jhb, 'Johannesburg Progress', last_n_months=8, color_map=color_map, stage_order=stage_order)
    
    # Plot the RP1 countries stacked bar chart
    melted_df = df_rp1_countries.melt(id_vars='Study site', value_vars=stage_order)
    print("Melted DataFrame for RP1 Countries")
    print(melted_df.head())
    fig = px.bar(melted_df, x='Study site', y='value', color='variable', title='Number of Studies in Relevant African Countries (RP1)',
                 color_discrete_map=color_map, barmode='stack')
    fig.update_layout(xaxis_title='Country', yaxis_title='Number of Studies')
    fig.update_traces(texttemplate='%{text:.2s}', textposition='inside')
    fig.update_xaxes(tickangle=-45)
    
    countries_fig = fig
    
    return rp1_fig, abj_fig, jhb_fig, countries_fig

if __name__ == '__main__':
    app.run_server(mode='inline')


RP1 DataFrame:
                                            Stage  Jan  Feb  Mar  Apr  May  \
0                Contact procedures not initiated  110  109  101   99   99   
1                              1st or 2nd invites   21   17   22   20   13   
2                             3rd or more invites   29   34   29   28   29   
3  Data sharing discussions and eligibility check   43   38   41   41   42   
4                                 DTA in progress   19   21   24   19   19   

   Jun  Jul  Aug  Sep  Oct  Nov  Dec  Jan.1  Feb.1  Mar.1  Apr.1  May.1  Jun.1  
0   99   99   96   96   96   82   74     73     57     58     58     37      0  
1    6    4    3    3    1   13   19     15     22     16     12     31     46  
2   34   24   18   15   15   15   17     18     19     23     25     25     33  
3   40   47   45   45   45   36   34     35     43     36     38     40     53  
4   19   24   21   24   25   25   25     28     24     31     26     24     21  

Abidjan Outputs DataFrame:
  

In [5]:
import pandas as pd
import geopandas as gpd
from dash import Dash, dcc, html
import dash_bootstrap_components as dbc
from dash.dependencies import Input, Output
import plotly.express as px

# Load the Excel file
file_path = 'Copy of HEAT_Tables_20.06.24_CPedits.xlsx'
xlsx_file = pd.read_excel(file_path, sheet_name=['RP1', 'Abj_outputs', 'Jhb_outputs', 'RP1_Countries'])

# Extract each DataFrame from the dictionary
df_rp1 = xlsx_file['RP1']
df_abj = xlsx_file['Abj_outputs']
df_jhb = xlsx_file['Jhb_outputs']
df_rp1_countries = xlsx_file['RP1_Countries']

# Define the color map and stage order to match the actual column names in df_rp1_countries
color_map = {
    '1st invite': '#1f77b4',
    '2nd invite': '#ff7f0e',
    '3rd or more invite': '#b8daff',
    'Interested': '#7f7f7f',
    'Follow up on interest': '#2ca02c',
    'Study documents received': '#d62728',
    'Eligibility confirmed': '#bcbd22',
    'Entering DTA': '#17becf',
    'DTA complete': '#9467bd',
    'Ineligible': '#8c564b',
    'Declined participation': '#ff9896',
    'Data unavailable': '#c5b0d5'
}
stage_order = [
    '1st invite', '2nd invite', '3rd or more invite', 'Interested',
    'Follow up on interest', 'Study documents received', 'Eligibility confirmed',
    'Entering DTA', 'DTA complete', 'Ineligible', 'Declined participation', 'Data unavailable'
]

# Convert the month columns in each DataFrame to a datetime format
def map_month_year(df, start_year=2023):
    month_map = {}
    encountered_dec = False  # Flag to indicate if we've encountered 'Dec'
    months_in_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    for column in df.columns:
        if any(month in column for month in months_in_order):
            base_month_name = ''.join(filter(str.isalpha, column))
            if base_month_name == 'Jan' and encountered_dec:
                start_year += 1
            if base_month_name == 'Dec':
                encountered_dec = True
            month_map[column] = f'{base_month_name} {start_year}'
    return df.rename(columns=month_map)

df_rp1 = map_month_year(df_rp1)
df_abj = map_month_year(df_abj)
df_jhb = map_month_year(df_jhb)

# Convert column names to datetime
def convert_column_to_datetime(df):
    new_columns = []
    for col in df.columns:
        if col == 'Stage':
            new_columns.append(col)
        else:
            date_str = col.strip() + ' 1'
            new_columns.append(pd.to_datetime(date_str, format='%b %Y %d', errors='coerce'))
    df.columns = new_columns

for dataframe in [df_rp1, df_abj, df_jhb]:
    convert_column_to_datetime(dataframe)

# Ensure the column names in df_rp1_countries match the stage_order
df_rp1_countries.columns = df_rp1_countries.columns.str.strip()  # Remove any leading/trailing whitespace

# Prepare df_rp1_countries for plotting
df_rp1_countries.set_index('Study site', inplace=True)
df_rp1_countries = df_rp1_countries[stage_order].fillna(0)
df_rp1_countries = df_rp1_countries[df_rp1_countries.sum(axis=1) > 0]
df_rp1_countries['Study site'] = df_rp1_countries.index

# Inspect the transformed data
print("\nTransformed RP1 DataFrame:")
print(df_rp1.head())
print("\nTransformed Abidjan Outputs DataFrame:")
print(df_abj.head())
print("\nTransformed Johannesburg Outputs DataFrame:")
print(df_jhb.head())
print("\nTransformed RP1 Countries DataFrame:")
print(df_rp1_countries.head())

# Function to plot the main stacked bar chart
def plot_stacked_bar_chart(df, title, last_n_months=8, color_map=None, stage_order=None):
    df = df.set_index('Stage').reindex(stage_order).fillna(0).reset_index()
    excluded_df = df[df['Stage'] == 'Ineligible/declined participation/data currently unavailable']
    df = df[df['Stage'] != 'Ineligible/declined participation/data currently unavailable']
    stages_df = df[~df['Stage'].str.contains("Total")]
    transposed_df = stages_df.set_index('Stage').transpose()
    transposed_df = transposed_df.iloc[-last_n_months:]

    # Debugging print statements
    print("Transposed DataFrame for", title)
    print(transposed_df.head())

    fig = px.bar(transposed_df, barmode='stack', title=title,
                 color_discrete_map=color_map)
    fig.update_layout(xaxis_title='Month', yaxis_title='Number of Studies')
    fig.update_traces(texttemplate='%{text:.2s}', textposition='inside')
    fig.update_xaxes(tickformat='%b %Y')
    return fig

# Create the app
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = html.Div([
    dbc.Row([
        dbc.Col(html.Div([
            dcc.Graph(id='rp1-bar-chart')
        ]), width=12)
    ]),
    dbc.Row([
        dbc.Col(html.Div([
            dcc.Graph(id='abj-bar-chart')
        ]), width=6),
        dbc.Col(html.Div([
            dcc.Graph(id='jhb-bar-chart')
        ]), width=6)
    ]),
    dbc.Row([
        dbc.Col(html.Div([
            dcc.Graph(id='rp1-countries-bar-chart')
        ]), width=12)
    ])
])

@app.callback(
    [Output('rp1-bar-chart', 'figure'),
     Output('abj-bar-chart', 'figure'),
     Output('jhb-bar-chart', 'figure'),
     Output('rp1-countries-bar-chart', 'figure')],
    [Input('rp1-bar-chart', 'id'),
     Input('abj-bar-chart', 'id'),
     Input('jhb-bar-chart', 'id'),
     Input('rp1-countries-bar-chart', 'id')]
)
def update_charts(*args):
    rp1_fig = plot_stacked_bar_chart(df_rp1, 'RP1 Progress', last_n_months=8, color_map=color_map, stage_order=stage_order)
    abj_fig = plot_stacked_bar_chart(df_abj, 'Abidjan Progress', last_n_months=8, color_map=color_map, stage_order=stage_order)
    jhb_fig = plot_stacked_bar_chart(df_jhb, 'Johannesburg Progress', last_n_months=8, color_map=color_map, stage_order=stage_order)
    
    # Plot the RP1 countries stacked bar chart
    melted_df = df_rp1_countries.melt(id_vars='Study site', value_vars=stage_order)
    print("Melted DataFrame for RP1 Countries")
    print(melted_df.head())
    fig = px.bar(melted_df, x='Study site', y='value', color='variable', title='Number of Studies in Relevant African Countries (RP1)',
                 color_discrete_map=color_map, barmode='stack')
    fig.update_layout(xaxis_title='Country', yaxis_title='Number of Studies')
    fig.update_traces(texttemplate='%{text:.2s}', textposition='inside')
    fig.update_xaxes(tickangle=-45)
    
    countries_fig = fig
    
    return rp1_fig, abj_fig, jhb_fig, countries_fig

if __name__ == '__main__':
    app.run_server(mode='inline')



Transformed RP1 DataFrame:
                                            Stage  2023-01-01 00:00:00  \
0                Contact procedures not initiated                  110   
1                              1st or 2nd invites                   21   
2                             3rd or more invites                   29   
3  Data sharing discussions and eligibility check                   43   
4                                 DTA in progress                   19   

   2023-02-01 00:00:00  2023-03-01 00:00:00  2023-04-01 00:00:00  \
0                  109                  101                   99   
1                   17                   22                   20   
2                   34                   29                   28   
3                   38                   41                   41   
4                   21                   24                   19   

   2023-05-01 00:00:00  2023-06-01 00:00:00  2023-07-01 00:00:00  \
0                   99                   99       

: 